In [2]:
import pandas as pd
import altair as alt

In [14]:
url = 'https://raw.githubusercontent.com/rgriff23/Olympic_history/refs/heads/master/data/athlete_events.csv'
alt.data_transformers.disable_max_rows()
df = pd.read_csv(url)
df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [15]:
df['Year'] = pd.to_datetime(df['Year'], format='%Y')

medal_counts = df[df['Medal'].notna()].groupby(['Year', 'NOC', 'Medal'])['Medal'].count().unstack(fill_value=0)
medal_counts['Total'] = medal_counts.sum(axis=1)
medal_counts = medal_counts.reset_index()

top_countries = medal_counts.groupby('NOC')['Total'].sum().nlargest(10).index
medal_counts_top10 = medal_counts[medal_counts['NOC'].isin(top_countries)]

In [16]:
base = alt.Chart(medal_counts_top10).encode(
    alt.X('Year:T'),
    alt.Y('Total:Q'),
    color = 'NOC:N'
)

lines = base.mark_line()

points = base.mark_point().encode(
    opacity = alt.value(0),
    tooltip = [
        alt.Tooltip('Year:T', title = 'Year'),
        alt.Tooltip('NOC:N', title = 'Country'),
        alt.Tooltip('Total:Q', title = 'Total Medals')
    ]
).add_params(
    alt.selection_point(nearest = True, on = 'mouseover', fields = ['Year'], empty = 'none')
)

country_selector = alt.selection_point(
    name = 'Country', fields = ['NOC'],
    bind = alt.binding_select(options = sorted(top_countries), name = "Select Country: ")
)

chart = (lines + points).properties(
    title='Medal Count by Country Over Time (Top 10 Countries)',
    width=800,
    height=400
).add_params(country_selector).transform_filter(
    country_selector
).interactive()

chart

alt.LayerChart(...)

In [17]:
myJekyllDir = '/Users/porsc/pjordan2.github.io/assets/json/'

In [18]:
chart.save(myJekyllDir+"altair_medal_count_final.json")

In [23]:
url2 = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRVt11gAvsQUN2916AT20BvCWGBz9GC2efYL1AFmqx1nxkIi8eo9onbPDSqlrvEaZjZ-sg2rluPY5aR/pub?gid=1658076489&single=true&output=csv'
alt.data_transformers.disable_max_rows()
df_forbes = pd.read_csv(url2)
df_forbes['Sport'] = df_forbes['Sport'].str.lower()
df_forbes.head()

,S.NO,Name,Nationality,Current Rank,Previous Year Rank,Sport,Year,earnings ($ million)
0,1,Mike Tyson,USA,1,NaN,boxing,1990,28.6
1,2,Buster Douglas,USA,2,NaN,boxing,1990,26.0
2,3,Sugar Ray Leonard,USA,3,NaN,boxing,1990,13.0
3,4,Ayrton Senna,Brazil,4,NaN,auto racing,1990,10.0
4,5,Alain Prost,France,5,NaN,auto racing,1990,9.0


In [24]:
top_sports = df_forbes.groupby('Sport')['earnings ($ million)'].sum().sort_values(ascending=False).head(10).reset_index()
top_sports = top_sports.sort_values('earnings ($ million)', ascending=True)

In [26]:
earnings_sport_base = alt.Chart(top_sports).encode(
    alt.X('earnings ($ million):Q', title='Total Earnings ($ million)'),
    alt.Y('Sport:N', sort='-x', title='')
)

line = earnings_sport_base.mark_rule().encode(
    x = 'earnings ($ million):Q'
)

point = earnings_sport_base.mark_circle(size=100).encode(
    alt.Color('Sport:N', legend=None)
)

text = earnings_sport_base.mark_text(
    align='left',
    baseline='middle',
    dx=5
).encode(
    text='earnings ($ million):Q'
)

earnings_sport_chart = (line + point + text).properties(
    title='Top 10 Sports by Total Athlete Earnings',
    width=600,
    height=300
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
)

earnings_sport_chart

alt.LayerChart(...)

In [27]:
myJekyllDir = '/Users/porsc/pjordan2.github.io/assets/json/'

In [28]:
earnings_sport_chart.save(myJekyllDir+"altair_sports_earnings.json")

In [29]:
top_nationalities = df_forbes.groupby('Nationality')['earnings ($ million)'].sum().sort_values(ascending=False).head(10).reset_index()

In [31]:
nationality_chart = alt.Chart(top_nationalities).mark_bar().encode(
    alt.X('earnings ($ million):Q', title = 'Total Earnings ($ million)'),
    alt.Y('Nationality:N', sort = '-x', title = ''),
    color = alt.Color('Nationality:N', legend = None),
    tooltip = ['Nationality', 'earnings ($ million)']
).properties(
    title = 'Top 10 Nationalities by Total Athlete Earnings',
    width=600,
    height=300
)

text = nationality_chart.mark_text(
    align = 'left',
    baseline = 'middle',
    dx = 3
).encode(
    text = 'earnings ($ million):Q'
)

final_chart = (nationality_chart + text).configure_axis(
    labelFontSize = 12,
    titleFontSize = 14
)

final_chart

alt.LayerChart(...)

In [32]:
myJekyllDir = '/Users/porsc/pjordan2.github.io/assets/json/'

In [33]:
final_chart.save(myJekyllDir+"altair_nationality_earnings.json")